<a href="https://colab.research.google.com/github/AnthonyMath1022/AnthonyMath1022/blob/main/Copy_of_Retail_Analysis_TFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive')

print('Loading data...')
train_path = '/content/drive/MyDrive/Reatilforecast/src/train.csv'
train_df = pd.read_csv(train_path, low_memory=True)
train_df['date'] = pd.to_datetime(train_df['date'])
pd.set_option('display.max_columns', 10)
display(train_df)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading data...


/tmp/ipykernel_2441/2959674660.py:9: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(train_path, low_memory=True)


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN
...,...,...,...,...,...,...
125497035,125497035,2017-08-15,54,2089339,4.0,False
125497036,125497036,2017-08-15,54,2106464,1.0,True
125497037,125497037,2017-08-15,54,2110456,192.0,False
125497038,125497038,2017-08-15,54,2113914,198.0,True


In [4]:
start_date = '2017-07-01'
end_date = '2017-12-31'

train_df = train_df[(train_df['date'] >= start_date) & (train_df['date'] <= end_date)]
pd.set_option('display.max_columns', 10)
display(train_df)
train_df.count()
train_df.isna().sum()


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
120642911,120642911,2017-07-01,1,99197,2.0,False
120642912,120642912,2017-07-01,1,103520,2.0,False
120642913,120642913,2017-07-01,1,103665,11.0,False
120642914,120642914,2017-07-01,1,105574,2.0,False
120642915,120642915,2017-07-01,1,105575,3.0,False
...,...,...,...,...,...,...
125497035,125497035,2017-08-15,54,2089339,4.0,False
125497036,125497036,2017-08-15,54,2106464,1.0,True
125497037,125497037,2017-08-15,54,2110456,192.0,False
125497038,125497038,2017-08-15,54,2113914,198.0,True


,0
id,0
date,0
store_nbr,0
item_nbr,0
unit_sales,0
onpromotion,0


In [5]:
import pandas as pd

item_path = '/content/drive/MyDrive/Reatilforecast/src/items.csv'
store_path = '/content/drive/MyDrive/Reatilforecast/src/stores.csv'
oil_path = '/content/drive/MyDrive/Reatilforecast/src/oil.csv'

item = pd.read_csv(item_path)
store = pd.read_csv(store_path)
oil = pd.read_csv(oil_path )
oil = oil[(oil['date'] >= start_date) & (oil['date'] <= end_date)]

train_df = pd.merge(train_df,item,on='item_nbr',how='left')
train_df = pd.merge(train_df,store,on='store_nbr',how='left')
train_df.isna().sum()

,0
id,0
date,0
store_nbr,0
item_nbr,0
unit_sales,0
onpromotion,0
family,0
class,0
perishable,0
city,0


In [6]:
import numpy as np

def most_frequent_sales(data, variable, N, all='TRUE'):


    # counts
    values, counts = np.unique(data[variable].to_numpy(), return_counts=True)

    # Ensure values and counts are 2D column vectors before stacking
    values_2d = values.reshape(-1, 1)
    counts_2d = counts.reshape(-1, 1)

    # Nx2 table: [value, count]
    labels_freq_pd = np.column_stack((values_2d, counts_2d))

    # sort by count desc
    labels_freq_pd = labels_freq_pd[np.argsort(labels_freq_pd[:, 1])[::-1]]

    # keep top N
    topN = labels_freq_pd[:N]
    main_labels = topN[:, 0] if all == 'False' else labels_freq_pd[:, 0]

    # raw labels (replace deprecated as_matrix)
    labels_raw_np = data[variable].to_numpy().reshape(-1, 1)

    # indices where label in main_labels (same style as your labels_filtered_index[0])
    labels_filtered_index = np.where(np.isin(labels_raw_np.ravel(), main_labels))

    return topN, labels_filtered_index

label_freq, labels_filtered_index = most_frequent_sales(train_df, 'item_nbr', 5, 'FALSE')
print("pd_train.shape=", labels_filtered_index[0].shape)

pd_train_filtered = train_df.loc[labels_filtered_index[0], :]
print("pd_train_filtered.shape = ", pd_train_filtered.shape)


pd_train.shape= (4854129,)
pd_train_filtered.shape =  (4854129, 13)


In [7]:
pd_train_filtered  = pd_train_filtered.drop(['city','state','type','cluster','store_nbr','item_nbr','family','class','id'], axis = 1)

In [8]:

dummy_variables = ['onpromotion','perishable']

for var in dummy_variables:
    dummy = pd.get_dummies(pd_train_filtered [var], prefix = var, drop_first = False).astype(int)
    pd_train_filtered  = pd.concat([pd_train_filtered ,dummy], axis = 1)

pd_train_filtered  = pd_train_filtered.drop(dummy_variables, axis = 1)


In [9]:
import pandas as pd

# 1. Ensure your date columns are actual datetime objects
pd_train_filtered ['date'] = pd.to_datetime(pd_train_filtered ['date'])
oil['date'] = pd.to_datetime(oil['date'])

# 2. Generate the complete date range automatically
# pd.date_range replaces the manual loop and delta calculation
calendar = pd.DataFrame({
    'date': pd.date_range(start=train_df.date.min(), end=train_df.date.max())
})

# 3. Merge
oil = calendar.merge(oil, on='date', how='left')

In [10]:
na_index_oil = oil[oil['dcoilwtico'].isnull() == True].index.values

#Define the index to use to apply the formala
na_index_oil_plus = na_index_oil.copy()
na_index_oil_minus = np.maximum(0, na_index_oil-1)

for i in range(len(na_index_oil)):
    k = 1
    while (na_index_oil[min(i+k,len(na_index_oil)-1)] == na_index_oil[i]+k):
        k += 1
    na_index_oil_plus[i] = min(len(oil)-1, na_index_oil_plus[i] + k )

#Apply the formula
for i in range(len(na_index_oil)):
    if (na_index_oil[i] == 0):
        oil.loc[na_index_oil[i], 'dcoilwtico'] = oil.loc[na_index_oil_plus[i], 'dcoilwtico']
    elif (na_index_oil[i] == len(oil)):
        oil.loc[na_index_oil[i], 'dcoilwtico'] = oil.loc[na_index_oil_minus[i], 'dcoilwtico']
    else:
        oil.loc[na_index_oil[i], 'dcoilwtico'] = (oil.loc[na_index_oil_plus[i], 'dcoilwtico'] + oil.loc[na_index_oil_minus[i], 'dcoilwtico'])/ 2

pd_train_filtered  = pd_train_filtered.merge(oil, left_on='date', right_on='date', how='left')


In [11]:
pd_train_filtered.sample(10)

,date,unit_sales,onpromotion_False,onpromotion_True,perishable_0,perishable_1,dcoilwtico
2276134,2017-07-22,2.476,1,0,0,1,45.9950
2407648,2017-07-23,3.000,1,0,1,0,46.1025
934904,2017-07-09,8.000,0,1,1,0,44.3625
4677118,2017-08-14,1.000,1,0,1,0,47.5900
219362,2017-07-02,1.000,1,0,1,0,45.1100
4287196,2017-08-10,1.181,1,0,0,1,48.5400
2454406,2017-07-24,7.000,1,0,1,0,46.2100
3839688,2017-08-06,7.000,1,0,1,0,49.4200
2078302,2017-07-20,8.000,1,0,1,0,46.7300
1545082,2017-07-15,2.000,1,0,1,0,46.2750


In [12]:
pd_train_filtered.count()

,0
date,4854129
unit_sales,4854129
onpromotion_False,4854129
onpromotion_True,4854129
perishable_0,4854129
perishable_1,4854129
dcoilwtico,4854129


In [13]:

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler

TOP_N = 10


# 1. Filter for Top 10 Items
# We use a wider date range (from Jan 2017) to ensure enough data for lookback and splitting.
local_start_date = '2017-01-01'

temp_base_df = pd.read_csv(train_path, low_memory=True)
temp_base_df['date'] = pd.to_datetime(temp_base_df['date'])
temp_base_df = temp_base_df[(temp_base_df['date'] >= local_start_date) & (temp_base_df['date'] <= end_date)].copy()

# Re-merge with item and store data
temp_item_df = pd.read_csv(item_path)
temp_store_df = pd.read_csv(store_path)
temp_base_df = pd.merge(temp_base_df, temp_item_df, on='item_nbr', how='left')
temp_base_df = pd.merge(temp_base_df, temp_store_df, on='store_nbr', how='left')

# Reload and merge Oil data (full range to cover Jan-July)
oil_full = pd.read_csv(oil_path)
oil_full['date'] = pd.to_datetime(oil_full['date'])
temp_base_df = pd.merge(temp_base_df, oil_full[['date', 'dcoilwtico']], on='date', how='left')

# Top 10 items
top_items = temp_base_df['item_nbr'].value_counts().head(TOP_N).index
pd_train_filtered = temp_base_df[temp_base_df['item_nbr'].isin(top_items)].copy()

# 2. Pivot the Data to Wide Format
# Sales Pivot
pivoted_sales = pd_train_filtered.pivot_table(
    index='date',
    columns='item_nbr',
    values='unit_sales',
    fill_value=0
)
pivoted_sales.columns = [f'sales_{i}' for i in pivoted_sales.columns]

# Promotion Pivot
if 'onpromotion' in pd_train_filtered.columns:
    pd_train_filtered['onpromotion'] = pd_train_filtered['onpromotion'].astype(int)
    pivoted_promo = pd_train_filtered.pivot_table(
        index='date',
        columns='item_nbr',
        values='onpromotion',
        fill_value=0
    )
    pivoted_promo.columns = [f'promo_{i}' for i in pivoted_promo.columns]
else:
    pivoted_promo = pd.DataFrame()

# Oil Data (Common Feature)
oil_series = pd_train_filtered.groupby('date')['dcoilwtico'].max()

# 3. Combine into one Wide Dataframe
combined = pd.concat([pivoted_sales, pivoted_promo, oil_series], axis=1)

# Fill gaps
combined = combined.ffill().bfill()

# Define targets and features
stock_cols = pivoted_sales.columns.tolist()
print(f"Target Columns ({len(stock_cols)}): {stock_cols}")
print(f"Total Features: {combined.shape[1]}")
print(f"Total Data Points (Days): {len(combined)}")



/tmp/ipykernel_2441/3124346341.py:13: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_base_df = pd.read_csv(train_path, low_memory=True)


Target Columns (10): ['sales_222879', 'sales_261052', 'sales_265559', 'sales_314384', 'sales_323013', 'sales_364606', 'sales_502331', 'sales_1052563', 'sales_1157564', 'sales_1162382']
Total Features: 21
Total Data Points (Days): 227


In [14]:
import pandas as pd
import numpy as np

# 1. Keep your filtered top 10 items data in long format
tft_df = pd_train_filtered[['date', 'item_nbr', 'unit_sales', 'onpromotion', 'dcoilwtico']].copy()

# 2. TFT requires an integer time index
tft_df = tft_df.sort_values(['item_nbr', 'date'])
time_mapping = {d: i for i, d in enumerate(tft_df['date'].unique())}
tft_df['time_idx'] = tft_df['date'].map(time_mapping)

# 3. Add categorical time features (TFT excels at using these)
tft_df['day_of_week'] = tft_df['date'].dt.dayofweek.astype(str)
tft_df['month'] = tft_df['date'].dt.month.astype(str)
tft_df['item_nbr'] = tft_df['item_nbr'].astype(str) # Identifiers must be strings/categories

# Fill NA in oil prices (forward fill then backward fill)
tft_df['dcoilwtico'] = tft_df.groupby('item_nbr')['dcoilwtico'].ffill().bfill()

In [15]:
!pip install pytorch-forecasting pytorch-lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.8/399.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 64.3 MB/s eta 0:00:00


In [16]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
import numpy as np
import pandas as pd

# Fix: Convert onpromotion to string so it can be used as a categorical feature
tft_df["onpromotion"] = tft_df["onpromotion"].astype(str)

# --- ROBUST CLEANING ---
# 1. Cast to float32 first to catch any values that might overflow when converting to tensors
tft_df['unit_sales'] = tft_df['unit_sales'].astype(np.float32)
tft_df['dcoilwtico'] = tft_df['dcoilwtico'].astype(np.float32)

# 2. Handle unit_sales (Target) using boolean indexing for robustness
# Replace Inf and NaN with 0.0
tft_df.loc[~np.isfinite(tft_df['unit_sales']), 'unit_sales'] = 0.0

# 3. Handle dcoilwtico (Feature)
# Fill missing values with forward/backward fill, then 0.0 as fallback
tft_df['dcoilwtico'] = tft_df['dcoilwtico'].ffill().bfill().fillna(0.0)
tft_df.loc[~np.isfinite(tft_df['dcoilwtico']), 'dcoilwtico'] = 0.0
# -----------------------

max_prediction_length = 7  # Predict 7 days out (your n_out)
max_encoder_length = 14    # Look back 14 days (your LOOKBACK)
training_cutoff = tft_df["time_idx"].max() - max_prediction_length

training = TimeSeriesDataSet(
    tft_df[tft_df.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="unit_sales",
    group_ids=["item_nbr"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["item_nbr"],
    time_varying_known_categoricals=["day_of_week", "month", "onpromotion"],
    time_varying_known_reals=["time_idx"],
    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=["unit_sales", "dcoilwtico"],
    target_normalizer=GroupNormalizer(
        groups=["item_nbr"]
    ),  # Normalizes each item individually
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True
)


validation = TimeSeriesDataSet.from_dataset(training, tft_df, predict=True, stop_randomization=True)

# Create dataloaders
batch_size = 64
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=2)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=2)

In [ ]:
import lightning.pytorch as pl
from pytorch_forecasting.models.temporal_fusion_transformer import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

# Define the TFT Model
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.001,
    hidden_size=32,          # Corresponds to your g_hidden
    attention_head_size=4,   # Number of attention heads
    dropout=0.1,
    hidden_continuous_size=16,
    loss=QuantileLoss(),     # TFT natively outputs confidence intervals (P10, P50, P90)
    optimizer="Adam"
)

# Train using PyTorch Lightning
trainer = pl.Trainer(
    max_epochs=30,
    accelerator="auto",
    gradient_clip_val=0.01,
)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    137 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    224 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  5.9 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  8.5 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  4.4 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    231 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 72.0 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 72.0 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 326                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

test_preds_original = np.zeros_like(test_preds)
test_targets_original = np.zeros_like(test_targets)

for i in range(n_out):
    # For each forecast step, inverse transform: [num_windows, N]
    test_preds_original[:, :, i] = scaler.inverse_transform(test_preds[:, :, i])
    test_targets_original[:, :, i] = scaler.inverse_transform(test_targets[:, :, i])

# For visualization, we can use the first forecast step or average across all steps
# Using the first forecast step (index 0):
inversed_actuals = test_targets_original[:, :, 0]
inversed_predictions = test_preds_original[:, :, 0]


plt.figure(figsize=(16, 12))
plt.suptitle('ChebConvTCN Predictions vs Actuals', fontsize=18, y=1.02)  # main title

for i, col in enumerate(stock_cols):
    plt.subplot(5, 3, i+1)
    plt.plot(inversed_actuals[:, i], label='Actual', color='blue')
    plt.plot(inversed_predictions[:, i], label='Predicted', color='red', linestyle='--')
    plt.title(f'{col} Closing Price')
    plt.xlabel('Time Step')
    plt.ylabel('Price')
    plt.legend()
    plt.grid(True)

plt.tight_layout(rect=[0, 0, 1, 0.97])  # leave room for suptitle
plt.show()


from sklearn import metrics
def mean_absolute_percentage_error(y_test, y_pred):
    y_true, y_pred = np.array(y_test), np.array(y_pred)
    return np.mean(np.abs((y_test - y_pred) / y_test)) * 100

def mean_squared_prediction_error(y_test, y_pred):
    y_true, y_pred = np.array(y_test), np.array(y_pred)
    return np.mean(((y_test - y_pred))**2)


for i, col in enumerate(stock_cols):
    print(f"\nMetrics for {col}:")
    print(f'MSE: {metrics.mean_squared_error(inversed_actuals[:, i], inversed_predictions[:, i])}')
    print(f'MAE: {metrics.mean_absolute_error(inversed_actuals[:, i], inversed_predictions[:, i])}')
    print(f'RMSE: {np.sqrt(metrics.mean_squared_error(inversed_actuals[:, i], inversed_predictions[:, i]))}')
    print(f'MAPE: {mean_absolute_percentage_error(inversed_actuals[:, i], inversed_predictions[:, i])}')
    print(f'MSPE: {mean_squared_prediction_error(inversed_actuals[:, i], inversed_predictions[:, i])}')
    print(f'sqrt MSPE: {np.sqrt(mean_squared_prediction_error(inversed_actuals[:, i], inversed_predictions[:, i]))}')
    print(f'R2: {r2_score(inversed_actuals[:, i], inversed_predictions[:, i])}')

# Overall metrics
print("\nOverall Metrics:")
print(f'MSE: {metrics.mean_squared_error(inversed_actuals, inversed_predictions)}')
print(f'MAE: {metrics.mean_absolute_error(inversed_actuals, inversed_predictions)}')
print(f'RMSE: {np.sqrt(metrics.mean_squared_error(inversed_actuals, inversed_predictions))}')
print(f'MAPE: {mean_absolute_percentage_error(inversed_actuals, inversed_predictions)}')
print(f'MSPE: {mean_squared_prediction_error(inversed_actuals, inversed_predictions)}')
print(f'sqrt MSPE: {np.sqrt(mean_squared_prediction_error(inversed_actuals, inversed_predictions))}')
print(f'R2: {r2_score(inversed_actuals, inversed_predictions)}')